In [3]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = Path.cwd().resolve()

print("Current working directory:")
print(PROJECT_ROOT)

required_paths = [
    "configs",
    "quran_asr",
    "scripts",
    "pyproject.toml",
]

missing = []

for path in required_paths:
    if not (PROJECT_ROOT / path).exists():
        missing.append(path)

if missing:
    raise RuntimeError(
        "Notebook belum dijalankan dari root project.\n"
        f"Posisi sekarang: {PROJECT_ROOT}\n"
        f"Missing: {missing}\n\n"
        "Pastikan Jupyter dibuka dari:\n"
        "/home/jrilym/Projects/AI/model-asr-quran"
    )

print("\nProject structure OK")
print("Python executable:", sys.executable)

Current working directory:
/home/jrilym/Projects/AI/model-asr-quran

Project structure OK
Python executable: /home/jrilym/Projects/AI/model-asr-quran/.venv/bin/python


In [4]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "local_3050.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Config belum ada: {CONFIG_PATH}\n"
        "Buat dulu configs/local_3050.yaml"
    )

print("Config ditemukan:")
print(CONFIG_PATH)

Config ditemukan:
/home/jrilym/Projects/AI/model-asr-quran/configs/local_3050.yaml


In [5]:
config_text = CONFIG_PATH.read_text(encoding="utf-8")

bad_patterns = [
    "/content",
    "/content/drive",
    "audio_dir: /data",
    "text_path: /data",
    "processed_dir: /data",
]

found = []

for pattern in bad_patterns:
    if pattern in config_text:
        found.append(pattern)

if found:
    raise RuntimeError(
        "Config masih mengandung path yang salah:\n"
        + "\n".join(f"- {item}" for item in found)
        + "\n\nGunakan path relatif seperti:\n"
        "audio_dir: data/raw/audio\n"
        "text_path: data/raw/text/quran_uthmani.json\n"
        "processed_dir: data/processed"
    )

print("Config path aman. Tidak ada /content atau /data absolute.")

Config path aman. Tidak ada /content atau /data absolute.


In [6]:
CACHE_DIR = PROJECT_ROOT / ".cache" / "hf"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "transformers")

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_HUB_CACHE:", os.environ["HF_HUB_CACHE"])
print("TRANSFORMERS_CACHE:", os.environ["TRANSFORMERS_CACHE"])

HF_HOME: /home/jrilym/Projects/AI/model-asr-quran/.cache/hf
HF_HUB_CACHE: /home/jrilym/Projects/AI/model-asr-quran/.cache/hf/hub
TRANSFORMERS_CACHE: /home/jrilym/Projects/AI/model-asr-quran/.cache/hf/transformers


In [7]:
print("NVIDIA SMI:")
subprocess.run(["nvidia-smi"], check=False)

print("\nDisk project:")
subprocess.run(["df", "-h", str(PROJECT_ROOT)], check=False)

NVIDIA SMI:
Wed Jul  8 09:39:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   67C    P0             55W /   60W |    2407MiB /   6144MiB |     73%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

CompletedProcess(args=['df', '-h', '/home/jrilym/Projects/AI/model-asr-quran'], returncode=0)

In [8]:
try:
    import torch

    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("VRAM GB:", round(props.total_memory / 1024**3, 2))
    else:
        print("Torch ada, tapi CUDA belum kebaca.")
except ImportError:
    print("Torch belum terinstall. Ini normal kalau belum menjalankan 01 setup.")

Torch: 2.12.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM GB: 5.67


In [9]:
paths_to_check = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "raw" / "audio",
    PROJECT_ROOT / "data" / "raw" / "text",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "artifacts",
    PROJECT_ROOT / "backups",
]

for path in paths_to_check:
    status = "OK" if path.exists() else "BELUM ADA"
    print(f"{status} - {path}")

OK - /home/jrilym/Projects/AI/model-asr-quran/data
OK - /home/jrilym/Projects/AI/model-asr-quran/data/raw
OK - /home/jrilym/Projects/AI/model-asr-quran/data/raw/audio
OK - /home/jrilym/Projects/AI/model-asr-quran/data/raw/text
OK - /home/jrilym/Projects/AI/model-asr-quran/data/processed
OK - /home/jrilym/Projects/AI/model-asr-quran/data/artifacts
OK - /home/jrilym/Projects/AI/model-asr-quran/backups


In [13]:
from pathlib import Path
import yaml

from datasets import load_from_disk

from quran_asr.tokenizer.processor import build_processor
from quran_asr.data_pipeline.normalize import normalize
from quran_asr.training.metrics import decode_labels_no_collapse

PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local_3050.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset = load_from_disk(str(PROJECT_ROOT / cfg["data"]["processed_dir"]))

processor = build_processor(
    PROJECT_ROOT / cfg["model"]["vocab_path"],
    cfg["data"]["sample_rate"],
)

split = "validation" if "validation" in dataset else "test"
text = dataset[split][0]["text"]
norm = normalize(text)
ids = processor.tokenizer(norm).input_ids
decoded = decode_labels_no_collapse(ids, processor)

print("SPLIT:", split)

print("\nTEXT:")
print(text)

print("\nNORMALIZED:")
print(norm)

print("\nDECODED LABEL:")
print(decoded)

print("\nUNK count:", ids.count(processor.tokenizer.unk_token_id))
print("num ids:", len(ids))

print("\npad_token:", processor.tokenizer.pad_token, processor.tokenizer.pad_token_id)
print("unk_token:", processor.tokenizer.unk_token, processor.tokenizer.unk_token_id)
print("vocab size:", len(processor.tokenizer))

SPLIT: validation

TEXT:
هَلْ أَتَىٰكَ حَدِيثُ ٱلْغَٰشِيَةِ

NORMALIZED:
هَلْ أَتَىٰكَ حَدِيثُ ٱلْغَٰشِيَةِ

DECODED LABEL:
هَلْ أَتَىٰكَ حَدِيثُ ٱلْغَٰشِيَةِ

UNK count: 0
num ids: 34

pad_token: [PAD] 49
unk_token: [UNK] 48
vocab size: 52
